# Multiple Linear Regression

## Libraries

TODO


FS
CV
Hyperparameter Tuning

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn import preprocessing
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df_train_y = pd.read_csv('data/Datasets_MLready/y_train.csv')
df_train_y.head()

,NO2,O3,PM10
0,17.20,44.33,18.81
1,9.33,49.89,16.57
2,12.80,46.59,27.35
3,24.10,41.80,19.33
4,8.67,50.66,11.73


In [4]:
df_train_X = pd.read_csv('data/Datasets_MLready/X_train_transformed.csv')
df_train_X.head()

,Month_sin__Month,Month_cos__Month,Day_sin__Day,Day_cos__Day,Weekday_sin__Weekday,Weekday_cos__Weekday,Time_sin__Time,Time_cos__Time,Spline_month__Month_sp_0,Spline_month__Month_sp_1,...,encoded__Location_Zch_Rosengartenstrasse,encoded__Location_Zch_Schimmelstrasse,encoded__Location_Zch_Stampfenbachstrasse,scaled__Hr,scaled__RainDur,scaled__T,scaled__WVv,scaled__StrGlo,scaled__meanVehicles,remainder__Year
0,-2.449294e-16,1.000000,0.433884,-0.900969,-0.781831,0.62349,-0.258819,0.965926,0.166667,0.666667,...,1.0,0.0,0.0,-0.106724,-0.347905,-0.442934,1.127894,-0.617743,-0.145789,2018.0
1,-2.449294e-16,1.000000,0.433884,-0.900969,-0.781831,0.62349,-0.258819,0.965926,0.166667,0.666667,...,0.0,1.0,0.0,-0.286880,-0.347905,-0.369828,-0.105097,-0.617743,-0.563898,2018.0
2,-2.449294e-16,1.000000,0.433884,-0.900969,-0.781831,0.62349,-0.258819,0.965926,0.166667,0.666667,...,0.0,0.0,1.0,-0.162810,-0.347905,-0.437893,4.097794,-0.617743,-0.864052,2018.0
3,5.000000e-01,0.866025,0.781831,0.623490,0.000000,1.00000,0.000000,1.000000,0.033083,0.529225,...,1.0,0.0,0.0,0.016779,-0.328655,-0.410163,0.849131,-0.617786,0.887373,2018.0
4,5.000000e-01,0.866025,0.781831,0.623490,0.000000,1.00000,0.000000,1.000000,0.033083,0.529225,...,0.0,1.0,0.0,-0.110690,-0.347905,-0.378651,0.205832,-0.617786,0.029038,2018.0


In [5]:
spline_cols = list(df_train_X.filter(like="Spline_").columns)
scaled_cols = list(df_train_X.filter(like="scaled_").columns)
encoded_cols = list(df_train_X.filter(like="encoded_").columns)
tri_cols = list(df_train_X.iloc[:,0:8].columns)

In [6]:
# plotting a pairplot
# df_pairplot = df.drop(columns=["Datetime", "Month", "Weekday", "YearMonth"])
#
# sns.set_theme(style="ticks")
#
# sns.pairplot(df_pairplot, hue="Location")

In [7]:
# df[["Month", "Weekday", "Year"]] = df[["Month", "Weekday", "Year"]].astype("category")
#
# df.dtypes

## 3. Time-based cross-validation

In [8]:
# fixed training set size
tss = TimeSeriesSplit(n_splits=5,gap=24*2,max_train_size=24 * 365,test_size=24*30)

## 4. Model Evaluation

In [9]:
# model evaluation
def evaluate(model, X, y):
    ret = cross_validate(model, X, y, cv=tss, scoring=["neg_root_mean_squared_error"])
    rmse = -ret["test_neg_root_mean_squared_error"]
    print("%.2f ± %.2f" % (rmse.mean(), rmse.std()))

In [10]:
X_tri = df_train_X[tri_cols + scaled_cols]
X_spl = df_train_X[spline_cols + scaled_cols]

In [11]:
y_NO2 = df_train_y["NO2"]
y_O3 = df_train_y["O3"]
y_PM10 = df_train_y["PM10"]

### Multiple Linear Regression

In [12]:
# building linear regression model
model_lr = LinearRegression()

In [13]:
print("linear regression, trigonometric encoding, NO2")
evaluate(model_lr, X_tri, y_NO2)

print("linear regression, spline encoding, NO2")
evaluate(model_lr, X_spl, y_NO2)

print("linear regression, trigonometric encoding, O3")
evaluate(model_lr, X_tri, y_O3)

print("linear regression, spline encoding, O3")
evaluate(model_lr, X_spl, y_O3)

print("linear regression, trigonometric encoding, PM10")
evaluate(model_lr, X_tri, y_PM10)

print("linear regression, spline encoding, PM10")
evaluate(model_lr, X_spl, y_PM10)

linear regression, trigonometric encoding, NO2
11.29 ± 3.47
linear regression, spline encoding, NO2
12.01 ± 3.82
linear regression, trigonometric encoding, O3
20.60 ± 3.63
linear regression, spline encoding, O3
38.89 ± 38.34
linear regression, trigonometric encoding, PM10
6.88 ± 1.41
linear regression, spline encoding, PM10
11.42 ± 9.91


In [22]:
model_lr.fit(X_tri, y_NO2)

LinearRegression()

In [23]:
feature_importance = pd.Series(
    np.abs(model_lr.coef_),
    index=X_tri.columns
).sort_values(ascending=False)
feature_importance

scaled__meanVehicles    7.046266
scaled__WVv             4.906478
Month_cos__Month        4.552908
Month_sin__Month        3.014353
Weekday_sin__Weekday    2.980195
scaled__StrGlo          1.871710
Weekday_cos__Weekday    1.648413
scaled__RainDur         1.186652
scaled__Hr              0.625669
Time_sin__Time          0.598818
Time_cos__Time          0.198336
Day_cos__Day            0.148323
scaled__T               0.141609
Day_sin__Day            0.052350
dtype: float64

### Gradient Boosting Regression

### Random Forest Regression

In [57]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    max_samples=0.5,   # each tree sees 50% of rows
    n_jobs=-1,
    random_state=42
)

rf.fit(X_tri, y_NO2)

RandomForestRegressor(max_depth=20, max_samples=0.5, min_samples_leaf=5,
                      n_estimators=200, n_jobs=-1, random_state=42)

In [50]:
df_test_X = pd.read_csv("data/Datasets_MLready/X_test_transformed.csv")
df_test_y = pd.read_csv("data/Datasets_MLready/y_test.csv")
y_NO2_test = df_test_y["NO2"]
X_tri_test = df_test_X[tri_cols + scaled_cols]

In [51]:
y_pred = rf.predict(X_tri_test)

rmse = (mean_squared_error(y_NO2_test, y_pred))**(1/2)
mae = mean_absolute_error(y_NO2_test, y_pred)
r2 = r2_score(y_NO2_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.2f}")

RMSE: 11.53
MAE: 9.21
R²: 0.28


In [52]:
feature_names = X_tri.columns

importances = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

importances

scaled__meanVehicles    0.322723
scaled__WVv             0.144956
scaled__T               0.086852
scaled__Hr              0.070545
Month_cos__Month        0.068962
Weekday_sin__Weekday    0.062987
scaled__StrGlo          0.047695
Time_sin__Time          0.047603
Month_sin__Month        0.035069
scaled__RainDur         0.032352
Time_cos__Time          0.031001
Day_sin__Day            0.023663
Weekday_cos__Weekday    0.013072
Day_cos__Day            0.012519
dtype: float64

## 5. Hyperparameter Tuning

### Multiple Linear Regression

### Gradient Boosting Regression

### Random Forest Regression

In [53]:
tss = TimeSeriesSplit(n_splits=5,gap=24*2,max_train_size=24 * 365,test_size=24*30)

param_dist = {
    "max_depth": [10, 15, 20, None],
    "min_samples_leaf": randint(1, 21),
    "min_samples_split": randint(2, 41),
    "max_features": ["sqrt", 0.5, 0.8],
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=25,                          # try 25 combos instead of all
    cv=tss,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=42
)

search.fit(X_tri, y_NO2)

RandomizedSearchCV(cv=TimeSeriesSplit(gap=48, max_train_size=8760, n_splits=5, test_size=720),
                   estimator=RandomForestRegressor(max_depth=20,
                                                   max_samples=0.5,
                                                   min_samples_leaf=5,
                                                   n_estimators=200, n_jobs=-1,
                                                   random_state=42),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'max_depth': [10, 15, 20, None],
                                        'max_features': ['sqrt', 0.5, 0.8],
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000001A280F1B170>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000001A280F1A540>},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [55]:
rf = RandomForestRegressor(max_depth=15, max_features='sqrt', max_samples=0.5,
                      min_samples_leaf=2, min_samples_split=21,
                      n_estimators=200, n_jobs=-1, random_state=42)

In [56]:
rf.fit(X_tri, y_NO2)

RandomForestRegressor(max_depth=15, max_features='sqrt', max_samples=0.5,
                      min_samples_leaf=2, min_samples_split=21,
                      n_estimators=200, n_jobs=-1, random_state=42)

In [54]:
X_tri_test = df_test_X[tri_cols + scaled_cols]
y_pred = rf.predict(X_tri_test)

rmse = (mean_squared_error(y_NO2_test, y_pred)) ** (1 / 2)
mae = mean_absolute_error(y_NO2_test, y_pred)
r2 = r2_score(y_NO2_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.2f}")

RMSE: 11.53
MAE: 9.21
R²: 0.28


## 6. Model Evaluation Metrics

averaged across CV splits during model selection
reported on the final test year (2025)
- RMSE
- MAE

## 7. Model Selection

Compare all models (e.g. linear regression, random forest, ...)

- Mean CV RMSE
- Stability across folds (standard deviation)
- Interpretability
- Computational complexity

## 8. Final Training and Testing

- The selected model is refit on all training data (2018–2024)
- Predictions are generated for 2025
- Final performance is reported on the test set

In [27]:
# fitting the model
model_lr.fit(X_tri, y_NO2)

LinearRegression()

In [28]:
df_test_X = pd.read_csv("data/Datasets_MLready/X_test_transformed.csv")
df_test_y = pd.read_csv("data/Datasets_MLready/y_test.csv")

In [33]:
y_NO2_test = df_test_y["NO2"]
y_NO2_test.describe()

count    25853.000000
mean        23.773432
std         13.584978
min          0.690000
25%         13.380000
50%         21.790000
75%         31.970000
max         95.510000
Name: NO2, dtype: float64

In [30]:
X_tri_test = df_test_X[encoded_cols + tri_cols + scaled_cols]
X_spl_test = df_test_X[spline_cols + tri_cols + scaled_cols]

In [31]:
predictions = model_lr.predict(X_tri_test)

In [32]:
# model evaluation
print('mean_squared_error:', mean_squared_error(y_NO2_test, predictions))
print('mean_absolute_error:', mean_absolute_error(y_NO2_test, predictions))

mean_squared_error: 144.96042374933566
mean_absolute_error: 9.889888824026036
